# Enterprise RAG — Hands-On, Part 6 of 11: Query transformation

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

### Setup — recap of state from earlier parts


In [ ]:
from enterprise_rag.identity import get_principal
from enterprise_rag.authz.policy import compile_prefilter
from enterprise_rag.llm.client import LLMClient
from enterprise_rag.ingest import store

llm = LLMClient()

# Build the index if it is not already there (same check as Part 4).
from enterprise_rag.ingest.store import collection_stats
try:
    stats = collection_stats("meridian")
    assert stats["chunks"] > 0
except Exception:
    from enterprise_rag.ingest.pipeline import ingest
    print("building index...")
    print(ingest().render())

---
# Part 6 - Query transformation

Three techniques, three different failure modes. Knowing *which* to reach for is the real skill;
running all three on every query is expensive theatre.

### 6.1 Multi-Query (= RAG-Fusion when you add RRF)

The user's wording is one sample from a space of phrasings. The corpus may use any other one.

In [ ]:
from enterprise_rag.retrieval import expansion

q = "my data is not showing up in dashboards"
variants = expansion.generate_multi_queries(llm, q)
for i, v in enumerate(variants):
    print(f"  {'[original]' if i == 0 else '[rewrite ]'} {v}")

### 6.2 HyDE - search with a *fabricated answer*

A question and its answer are written in different registers. `"Why did ingest stall?"` embeds
nowhere near `"the compaction queue saturated at 08:47"`.

So have the model **invent** a plausible answer and embed *that* instead. The invention is never
shown to anyone - it is only a search probe.

In [ ]:
passage = expansion.generate_hyde_passage(llm, "why did ingest stall in March?")
print("HYPOTHETICAL PASSAGE (invented, used only as a search probe):\n")
print(textwrap.fill(passage, 92))

In [ ]:
# Does the fabricated passage actually retrieve better than the raw question?
marco = get_principal("u_marco_t3")
w_marco = compile_prefilter(marco)
question = "why did ingest stall in March?"

for label, text in [("raw question", question), ("HyDE passage", passage)]:
    v = llm.embed([text])[0]
    print(f"question is {text}")
    hits = store.dense_search(marco.tenant_id, v, w_marco, 4)
    print(f"{label:<15} -> " + ", ".join(f"{h.chunk.chunk_id}({h.score:.3f})" for h in hits))

### 6.3 Decomposition - multi-hop questions

No single chunk answers *"why did they lose data **and** what does their contract promise?"*.

In [ ]:
subs = expansion.decompose(llm, "Why did Vertex Financial lose data on 14 March "
                                "and what does their contract entitle them to?")
for s in subs:
    print("  -", s)

print()
single = expansion.decompose(llm, "What does MRD-5031 mean?")
print("single-hop question ->", single or "no decomposition needed (correct)")

---

**◀ Previous:** [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb)

**Next ▶:** [7. Reranking](part07-reranking.ipynb)
